In [2]:
import cv2
import glob
import os
import numpy as np

# Paths
original_train_images_dir = "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/BCCD/images/train/"
percentile_clipped_dir = "/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/ClipPercentile/images/train/"

print(f"Verified original_train_images_dir: {original_train_images_dir}")
print(f"Verified percentile_clipped_dir: {percentile_clipped_dir}")

# Create output directory
os.makedirs(percentile_clipped_dir, exist_ok=True)

# Percentile clipping parameters (adjust based on your dataset)
LOWER_PERCENTILE = 1.0   # Clip darkest 1%
UPPER_PERCENTILE = 99.0  # Clip brightest 1%

# Get all image paths (multiple extensions)
image_paths = []
for ext in ['*.jpg']:
    image_paths.extend(glob.glob(os.path.join(original_train_images_dir, ext)))

print(f"Attempting to process {len(image_paths)} training images...")

if not image_paths:
    print(f"No image files found in {original_train_images_dir}")
    print("Please ensure the path is correct and images have .jpg extensions.")
else:
    processed_count = 0
    for img_path in image_paths:
        # Read color image (BGR)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not read image {img_path}. Skipping.")
            continue

        # Convert to LAB color space (L = lightness, A/B = color channels)
        lab_img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab_img)

        # Apply percentile clipping ONLY to L channel (preserves color balance)
        p_lower = np.percentile(l_channel, LOWER_PERCENTILE)
        p_upper = np.percentile(l_channel, UPPER_PERCENTILE)

        # Clip and rescale L channel
        clipped_l = np.clip(l_channel, p_lower, p_upper)
        if p_upper > p_lower:
            clipped_l = ((clipped_l - p_lower) / (p_upper - p_lower) * 255).astype(np.uint8)
        else:
            clipped_l = np.zeros_like(l_channel, dtype=np.uint8)  # Uniform image edge case

        # Merge channels back
        clipped_lab = cv2.merge([clipped_l, a_channel, b_channel])

        # Convert back to BGR
        clipped_bgr = cv2.cvtColor(clipped_lab, cv2.COLOR_LAB2BGR)

        # Save result
        file_name = os.path.basename(img_path)
        output_path = os.path.join(percentile_clipped_dir, file_name)
        cv2.imwrite(output_path, clipped_bgr)

        processed_count += 1

    print(f"✓ Successfully processed {processed_count}/{len(image_paths)} images")
    print(f"✓ Color-preserving percentile clipping completed")
    print(f"✓ Output directory: {percentile_clipped_dir}")
    print(f"\n💡 Why LAB color space?")
    print("   - L channel controls brightness/contrast (where we apply clipping)")
    print("   - A/B channels preserve original color information (critical for blood cells)")
    print("   - Prevents color shifts that occur with per-channel clipping")

Verified original_train_images_dir: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/BCCD/images/train/
Verified percentile_clipped_dir: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/ClipPercentile/images/train/
Attempting to process 327 training images...
✓ Successfully processed 327/327 images
✓ Color-preserving percentile clipping completed
✓ Output directory: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/CellDetection/ClipPercentile/images/train/

💡 Why LAB color space?
   - L channel controls brightness/contrast (where we apply clipping)
   - A/B channels preserve original color information (critical for blood cells)
   - Prevents color shifts that occur with per-channel clipping
